# 1. Producing the data  
In this task, we will implement Apache Kafka producers to simulate real-time data streaming. <b>Spark and parallel data processing should not be used in this section, as we are simulating sensors that often lack processing capabilities.</b>  

1.	Every 5 seconds, load 5 days of weather data from the CSV file. We refer to this as weather5s to explain the tasks; feel free to use your own variable name. You should keep a pointer in the file reading process and advance it per read. The data reading should be in chronological order.
2.	Add the current timestamp (weather_ts) to the weather5s and spread your batch out evenly for 5 seconds for each day. Since the weather data is hourly readings, each day you shall have 24 records (120 records in total for 5 days).
For example, assume you send the records at 2025-01-26 00:00:00 (ISO format: YYYY-MM-DD HH:MM:SS) -> (ts = 1737810000):  
Day 1(records 1-24): ts = 1737810000  
Day 2(records 25-48): ts = 1737810001  
Day 3(records 49-72): ts = 1737810002  
…
3.	Send your batch of weather data to a Kafka topic with an appropriate name.




In [ ]:
from kafka3 import KafkaProducer
from json import dumps
import datetime as dt
import csv
import time

hostip = 'kafka' #dear tutor, change here if needed.

def read_weather_csv(file):
    list = []
    with open(file, 'rt') as f:
        reader = csv.DictReader(f)
        for row in reader:
            list.append(row)
    return list
        
def connect_kafka_producer():
    _producer = None
    try:
        _producer = KafkaProducer(bootstrap_servers=[f'{hostip}:9092'],
                                  value_serializer=lambda x: dumps(x).encode('ascii'),
                                  api_version=(0, 10))
    except Exception as ex:
        print('Exception while connecting Kafka.')
        print(str(ex))
    finally:
        return _producer

if __name__ == '__main__':
    topic = 'weather5s'
    data = read_weather_csv('weather.csv')
    
    producer = connect_kafka_producer()
    print("Streaming weather data...")

    start_index = 0
    rows_per_batch = 120  # 5 days * 24 hours

    while True:
        # get next 120 records
        batch = data[start_index:start_index + rows_per_batch]
        start_index += rows_per_batch

        # base timestamp
        base_ts = int(dt.datetime.now().timestamp())

        # assign timestamps (each day = +1 second)
        for i, row in enumerate(batch):
            ts_offset = i // 24  # changes every 24 records
            row['weather_ts'] = base_ts + ts_offset

        producer.send(topic, batch)
        print(f"Sent {len(batch)} records starting at ts={base_ts}")
        print("First five rows: ", batch[0:5]) # print 5 records of the 120 just to visualise
        print("----------------------")
        
        # ensure infinite loop. When the file reaches end, it will restart from the top.
        if start_index >= len(data):
            start_index = 0

        time.sleep(5)


Streaming weather data...
Sent 120 records starting at ts=1761459627
First five rows:  [{'site_id': '0', 'timestamp': '2022-01-01 22:00:00.000', 'air_temperature': '26.7', 'cloud_coverage': '', 'dew_temperature': '18.3', 'sea_level_pressure': '1016.9', 'wind_direction': '230.0', 'wind_speed': '3.1', 'weather_ts': 1761459627}, {'site_id': '0', 'timestamp': '2022-01-01 23:00:00.000', 'air_temperature': '25.6', 'cloud_coverage': '', 'dew_temperature': '18.3', 'sea_level_pressure': '1017.5', 'wind_direction': '230.0', 'wind_speed': '3.1', 'weather_ts': 1761459627}, {'site_id': '0', 'timestamp': '2022-01-02 00:00:00.000', 'air_temperature': '24.4', 'cloud_coverage': '6.0', 'dew_temperature': '18.9', 'sea_level_pressure': '1018.1', 'wind_direction': '270.0', 'wind_speed': '2.6', 'weather_ts': 1761459627}, {'site_id': '0', 'timestamp': '2022-01-02 01:00:00.000', 'air_temperature': '23.9', 'cloud_coverage': '4.0', 'dew_temperature': '18.3', 'sea_level_pressure': '1018.5', 'wind_direction': '30

Sent 120 records starting at ts=1761459661
First five rows:  [{'site_id': '0', 'timestamp': '2022-02-05 22:00:00.000', 'air_temperature': '13.9', 'cloud_coverage': '2.0', 'dew_temperature': '1.7', 'sea_level_pressure': '1023.7', 'wind_direction': '10.0', 'wind_speed': '9.3', 'weather_ts': 1761459661}, {'site_id': '0', 'timestamp': '2022-02-05 23:00:00.000', 'air_temperature': '12.8', 'cloud_coverage': '2.0', 'dew_temperature': '4.4', 'sea_level_pressure': '1024.3', 'wind_direction': '10.0', 'wind_speed': '6.7', 'weather_ts': 1761459661}, {'site_id': '0', 'timestamp': '2022-02-06 00:00:00.000', 'air_temperature': '11.7', 'cloud_coverage': '0.0', 'dew_temperature': '5.6', 'sea_level_pressure': '1024.5', 'wind_direction': '10.0', 'wind_speed': '5.7', 'weather_ts': 1761459661}, {'site_id': '0', 'timestamp': '2022-02-06 01:00:00.000', 'air_temperature': '11.1', 'cloud_coverage': '0.0', 'dew_temperature': '6.1', 'sea_level_pressure': '1024.8', 'wind_direction': '360.0', 'wind_speed': '5.1', 

Sent 120 records starting at ts=1761459695
First five rows:  [{'site_id': '0', 'timestamp': '2022-03-11 22:00:00.000', 'air_temperature': '26.7', 'cloud_coverage': '', 'dew_temperature': '17.8', 'sea_level_pressure': '1020.8', 'wind_direction': '100.0', 'wind_speed': '6.2', 'weather_ts': 1761459695}, {'site_id': '0', 'timestamp': '2022-03-11 23:00:00.000', 'air_temperature': '25.6', 'cloud_coverage': '', 'dew_temperature': '17.8', 'sea_level_pressure': '1021.8', 'wind_direction': '80.0', 'wind_speed': '4.1', 'weather_ts': 1761459695}, {'site_id': '0', 'timestamp': '2022-03-12 00:00:00.000', 'air_temperature': '23.3', 'cloud_coverage': '4.0', 'dew_temperature': '17.8', 'sea_level_pressure': '1022.0', 'wind_direction': '90.0', 'wind_speed': '3.1', 'weather_ts': 1761459695}, {'site_id': '0', 'timestamp': '2022-03-12 01:00:00.000', 'air_temperature': '21.7', 'cloud_coverage': '2.0', 'dew_temperature': '17.8', 'sea_level_pressure': '1022.4', 'wind_direction': '100.0', 'wind_speed': '3.1', '

Sent 120 records starting at ts=1761459729
First five rows:  [{'site_id': '0', 'timestamp': '2022-04-15 22:00:00.000', 'air_temperature': '25.6', 'cloud_coverage': '', 'dew_temperature': '18.9', 'sea_level_pressure': '1014.1', 'wind_direction': '40.0', 'wind_speed': '7.2', 'weather_ts': 1761459729}, {'site_id': '0', 'timestamp': '2022-04-15 23:00:00.000', 'air_temperature': '23.3', 'cloud_coverage': '', 'dew_temperature': '18.9', 'sea_level_pressure': '1014.6', 'wind_direction': '50.0', 'wind_speed': '8.8', 'weather_ts': 1761459729}, {'site_id': '0', 'timestamp': '2022-04-16 00:00:00.000', 'air_temperature': '22.2', 'cloud_coverage': '6.0', 'dew_temperature': '18.9', 'sea_level_pressure': '1015.2', 'wind_direction': '70.0', 'wind_speed': '6.2', 'weather_ts': 1761459729}, {'site_id': '0', 'timestamp': '2022-04-16 01:00:00.000', 'air_temperature': '21.7', 'cloud_coverage': '', 'dew_temperature': '18.9', 'sea_level_pressure': '1015.9', 'wind_direction': '60.0', 'wind_speed': '5.7', 'weath

Sent 120 records starting at ts=1761459763
First five rows:  [{'site_id': '0', 'timestamp': '2022-05-20 22:00:00.000', 'air_temperature': '27.8', 'cloud_coverage': '', 'dew_temperature': '20.0', 'sea_level_pressure': '1016.1', 'wind_direction': '100.0', 'wind_speed': '5.7', 'weather_ts': 1761459763}, {'site_id': '0', 'timestamp': '2022-05-20 23:00:00.000', 'air_temperature': '27.2', 'cloud_coverage': '', 'dew_temperature': '19.4', 'sea_level_pressure': '1016.7', 'wind_direction': '0.0', 'wind_speed': '0.0', 'weather_ts': 1761459763}, {'site_id': '0', 'timestamp': '2022-05-21 00:00:00.000', 'air_temperature': '26.7', 'cloud_coverage': '6.0', 'dew_temperature': '19.4', 'sea_level_pressure': '1017.5', 'wind_direction': '0.0', 'wind_speed': '0.0', 'weather_ts': 1761459763}, {'site_id': '0', 'timestamp': '2022-05-21 01:00:00.000', 'air_temperature': '25.6', 'cloud_coverage': '', 'dew_temperature': '20.6', 'sea_level_pressure': '1017.9', 'wind_direction': '0.0', 'wind_speed': '0.0', 'weather

...

Sent 120 records starting at ts=1761466088
First five rows:  [{'site_id': '2', 'timestamp': '2022-06-13 18:00:00.000', 'air_temperature': '32.8', 'cloud_coverage': '2.0', 'dew_temperature': '-2.8', 'sea_level_pressure': '1012.5', 'wind_direction': '', 'wind_speed': '2.6', 'weather_ts': 1761466088}, {'site_id': '2', 'timestamp': '2022-06-13 19:00:00.000', 'air_temperature': '33.3', 'cloud_coverage': '2.0', 'dew_temperature': '-3.3', 'sea_level_pressure': '1011.9', 'wind_direction': '0.0', 'wind_speed': '0.0', 'weather_ts': 1761466088}, {'site_id': '2', 'timestamp': '2022-06-13 20:00:00.000', 'air_temperature': '36.1', 'cloud_coverage': '2.0', 'dew_temperature': '-2.8', 'sea_level_pressure': '1011.1', 'wind_direction': '280.0', 'wind_speed': '5.1', 'weather_ts': 1761466088}, {'site_id': '2', 'timestamp': '2022-06-13 21:00:00.000', 'air_temperature': '36.1', 'cloud_coverage': '2.0', 'dew_temperature': '-2.8', 'sea_level_pressure': '1010.5', 'wind_direction': '280.0', 'wind_speed': '4.1', 

Sent 120 records starting at ts=1761466121
First five rows:  [{'site_id': '2', 'timestamp': '2022-07-18 18:00:00.000', 'air_temperature': '35.0', 'cloud_coverage': '6.0', 'dew_temperature': '16.1', 'sea_level_pressure': '1013.5', 'wind_direction': '0.0', 'wind_speed': '0.0', 'weather_ts': 1761466121}, {'site_id': '2', 'timestamp': '2022-07-18 19:00:00.000', 'air_temperature': '38.3', 'cloud_coverage': '4.0', 'dew_temperature': '16.1', 'sea_level_pressure': '1012.8', 'wind_direction': '180.0', 'wind_speed': '4.6', 'weather_ts': 1761466121}, {'site_id': '2', 'timestamp': '2022-07-18 20:00:00.000', 'air_temperature': '37.8', 'cloud_coverage': '4.0', 'dew_temperature': '15.6', 'sea_level_pressure': '1012.1', 'wind_direction': '0.0', 'wind_speed': '0.0', 'weather_ts': 1761466121}, {'site_id': '2', 'timestamp': '2022-07-18 21:00:00.000', 'air_temperature': '38.3', 'cloud_coverage': '4.0', 'dew_temperature': '15.6', 'sea_level_pressure': '1011.1', 'wind_direction': '250.0', 'wind_speed': '2.1

Sent 120 records starting at ts=1761466155
First five rows:  [{'site_id': '2', 'timestamp': '2022-08-22 18:00:00.000', 'air_temperature': '33.3', 'cloud_coverage': '2.0', 'dew_temperature': '15.0', 'sea_level_pressure': '1010.8', 'wind_direction': '50.0', 'wind_speed': '1.5', 'weather_ts': 1761466155}, {'site_id': '2', 'timestamp': '2022-08-22 19:00:00.000', 'air_temperature': '33.9', 'cloud_coverage': '2.0', 'dew_temperature': '14.4', 'sea_level_pressure': '1010.0', 'wind_direction': '300.0', 'wind_speed': '2.1', 'weather_ts': 1761466155}, {'site_id': '2', 'timestamp': '2022-08-22 20:00:00.000', 'air_temperature': '35.6', 'cloud_coverage': '4.0', 'dew_temperature': '13.9', 'sea_level_pressure': '1009.2', 'wind_direction': '290.0', 'wind_speed': '3.1', 'weather_ts': 1761466155}, {'site_id': '2', 'timestamp': '2022-08-22 21:00:00.000', 'air_temperature': '37.2', 'cloud_coverage': '4.0', 'dew_temperature': '13.9', 'sea_level_pressure': '1008.0', 'wind_direction': '240.0', 'wind_speed': '

Sent 120 records starting at ts=1761466188
First five rows:  [{'site_id': '2', 'timestamp': '2022-09-26 19:00:00.000', 'air_temperature': '31.1', 'cloud_coverage': '', 'dew_temperature': '8.3', 'sea_level_pressure': '1015.7', 'wind_direction': '150.0', 'wind_speed': '5.7', 'weather_ts': 1761466188}, {'site_id': '2', 'timestamp': '2022-09-26 20:00:00.000', 'air_temperature': '30.6', 'cloud_coverage': '', 'dew_temperature': '7.8', 'sea_level_pressure': '1014.8', 'wind_direction': '140.0', 'wind_speed': '5.7', 'weather_ts': 1761466188}, {'site_id': '2', 'timestamp': '2022-09-26 21:00:00.000', 'air_temperature': '31.7', 'cloud_coverage': '', 'dew_temperature': '8.9', 'sea_level_pressure': '1013.8', 'wind_direction': '140.0', 'wind_speed': '6.7', 'weather_ts': 1761466188}, {'site_id': '2', 'timestamp': '2022-09-26 22:00:00.000', 'air_temperature': '31.1', 'cloud_coverage': '', 'dew_temperature': '8.9', 'sea_level_pressure': '1013.7', 'wind_direction': '120.0', 'wind_speed': '3.1', 'weather_

Sent 120 records starting at ts=1761466221
First five rows:  [{'site_id': '2', 'timestamp': '2022-10-31 19:00:00.000', 'air_temperature': '27.8', 'cloud_coverage': '', 'dew_temperature': '8.3', 'sea_level_pressure': '1011.4', 'wind_direction': '0.0', 'wind_speed': '0.0', 'weather_ts': 1761466221}, {'site_id': '2', 'timestamp': '2022-10-31 20:00:00.000', 'air_temperature': '28.9', 'cloud_coverage': '', 'dew_temperature': '8.9', 'sea_level_pressure': '1010.2', 'wind_direction': '0.0', 'wind_speed': '0.0', 'weather_ts': 1761466221}, {'site_id': '2', 'timestamp': '2022-10-31 21:00:00.000', 'air_temperature': '28.9', 'cloud_coverage': '', 'dew_temperature': '9.4', 'sea_level_pressure': '1009.2', 'wind_direction': '240.0', 'wind_speed': '1.5', 'weather_ts': 1761466221}, {'site_id': '2', 'timestamp': '2022-10-31 22:00:00.000', 'air_temperature': '29.4', 'cloud_coverage': '', 'dew_temperature': '10.6', 'sea_level_pressure': '1008.7', 'wind_direction': '', 'wind_speed': '2.1', 'weather_ts': 176